# Lesson 16 Lab — TensorFlow MOT and the Keras Pruning/Export Lifecycle

**Puzzle:** Why can training-time Keras sparsity fail to reduce a deployable TFLite artifact?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

TensorFlow Model Optimization wraps Keras layers with masks, thresholds, and pruning-step state. Export requires updating the pruning step during training, reaching the schedule, stripping wrappers, converting, and checking the final representation. This environment may not provide TensorFlow, so availability is an explicit result rather than an excuse for invented output.


## 0. Predict before running

1. Predict the target sparsity before, during, and after a polynomial window.
2. Predict what `strip_pruning` removes and what it retains.
3. List the artifacts required before claiming a TFLite size benefit.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The lab records TensorFlow and TFMOT package availability, evaluates the polynomial schedule formula on CUDA, constructs the required lifecycle state machine, and runs a tiny native strip/export probe only when dependencies exist.

- Schedule progress depends on optimizer steps and update callbacks.
- Stripping removes training wrappers, not necessarily dense storage.
- TensorFlow/TFMOT availability is part of reproducible backend evidence.


## 2. Derive the mechanism

Polynomial decay controls target sparsity by optimizer step. Wrappers contain training-only variables and callbacks update their step. `strip_pruning` removes wrappers while retaining sparse weights; it does not guarantee a smaller uncompressed format or a sparse-accelerated runtime. TFLite conversion and optional compression are separate gates. A native experiment must therefore preserve versions, wrapper state, stripped model, converted bytes, and output parity.

### Mechanism at a glance

```mermaid
stateDiagram-v2
  [*] --> DenseKeras
  DenseKeras --> Wrapped: prune_low_magnitude
  Wrapped --> Scheduled: training + pruning-step updates
  Scheduled --> Stripped: strip_pruning
  Stripped --> Exported: SavedModel / TFLite conversion
  Exported --> Verified: load, size, quality, runtime checks
```

### Walk it step by step

1. **Wrap the model before training.** The pruning wrapper owns masks and schedule state; it is not equivalent to a permanently smaller Keras layer.
2. **Advance the pruning step.** Callbacks or explicit updates must keep the schedule synchronized with optimizer steps.
3. **Strip training-only wrappers.** After training, materialize the masked weights and remove wrapper state before export.
4. **Verify the deployment artifact.** Load the stripped model, convert to the target format, and measure compressed size and runtime behavior separately.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 16
LESSON_TITLE = 'TensorFlow MOT and the Keras Pruning/Export Lifecycle'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260824
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | CUDA-evaluated polynomial schedule and an unstripped lifecycle state |
| Candidate | native TFMOT wrapper/strip probe when available, otherwise a bounded compatibility result |
| Held constant | environment, schedule endpoints, steps, target rate, lifecycle transitions, and seed |
| Measurements | package availability, schedule values, native probe status, and lifecycle gates |
| Evidence | `compatibility-probe` |

**Experiment:** Probe the Keras pruning stack and execute the schedule/lifecycle contract without fabricating a missing native backend.


## 5. Read the experiment code

The notebook uses the published polynomial form to produce deterministic target rates and checks that strip/export cannot be marked complete before training and wrapper removal. Conditional imports keep missing packages in a structured field. No Keras latency or TFLite size number is synthesized when the stack is absent.

Do not execute until the code implements the frozen table above.


In [2]:
tf_available=importlib.util.find_spec("tensorflow") is not None; tfmot_available=importlib.util.find_spec("tensorflow_model_optimization") is not None
initial,final,begin,end=0.0,0.80,10,50
def schedule(step):
    if step<=begin:return initial
    if step>=end:return final
    p=(step-begin)/(end-begin); return final+(initial-final)*(1-p)**3
steps=torch.tensor([0,10,20,30,40,50,60],device=DEVICE); schedule_values=[float(schedule(int(s))) for s in steps.cpu().tolist()]
native=False; native_message="TensorFlow/TFMOT not both available"
if tf_available and tfmot_available:
    try:
        import tensorflow as tf, tensorflow_model_optimization as tfmot
        base=tf.keras.Sequential([tf.keras.layers.Input((8,)),tf.keras.layers.Dense(4)])
        wrapped=tfmot.sparsity.keras.prune_low_magnitude(base); stripped=tfmot.sparsity.keras.strip_pruning(wrapped); native=True; native_message=stripped.__class__.__name__
    except Exception as exc:native_message=f"{type(exc).__name__}: {str(exc).splitlines()[0]}"
metrics={"tensorflow_available":tf_available,"tfmot_available":tfmot_available,"schedule_steps":steps.cpu().tolist(),"schedule_values":schedule_values,"mid_schedule_sparsity":schedule(30),"final_schedule_sparsity":schedule(50),"native_probe_executed":native,"native_message":native_message,"lifecycle_ready":bool(native)}
analysis=(f"The cubic schedule moved from {schedule_values[1]:.1%} at step 10 to {metrics['mid_schedule_sparsity']:.1%} "
          f"at step 30 and {metrics['final_schedule_sparsity']:.1%} at step 50. TensorFlow/TFMOT availability was "
          f"{tf_available}/{tfmot_available}, so native wrapper stripping executed={native}. Missing native stages remain false rather than inferred.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| TensorFlow available | no |
| TFMOT available | no |
| Mid-schedule sparsity | 70.00% |
| Final schedule sparsity | 80.00% |
| Native probe executed | no |
| Lifecycle ready | no |


## 7. Interpret rather than merely print

The cubic schedule moved from 0.0% at step 10 to 70.0% at step 30 and 80.0% at step 50. TensorFlow/TFMOT availability was False/False, so native wrapper stripping executed=False. Missing native stages remain false rather than inferred.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The notebook records real package/API availability and preserves the native success or failure state. Missing backend execution remains unmeasured.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 16,
    "title": 'TensorFlow MOT and the Keras Pruning/Export Lifecycle',
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Keras pruning is a versioned train-strip-convert lifecycle; a missing native stack must remain visibly unexecuted.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 16,
  "title": "TensorFlow MOT and the Keras Pruning/Export Lifecycle",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260824
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "tensorflow_available": false,
    "tfmot_available": false,
    "schedule_steps": [
      0,
      10,
      20,
      30,
      40,
      50,
      60
    ],
    "schedule_values": [
      0.0,
      0.0,
      0.4625,
      0.7000000000000001,
      0.7875000000000001,
      0.8,
      0.8
    ],
    "mid_schedule_sparsity": 0.7000000000000001,
    "final_schedule_sparsity": 0.8,
    "native_probe_executed": false,
    "native_message": "TensorFlow/TFMOT not both available",
    "lifecycle_ready": false
  },
  "analysis": "The cubic schedule moved from 0.0% at step 10 to 70.0% at step 30 and 80.0% at step 50. TensorFlow/TFMOT availability was False

## 9. Make the bounded decision

> Keras pruning is a versioned train-strip-convert lifecycle; a missing native stack must remain visibly unexecuted.

**Acceptance/rollback:** Accept a Keras pruning delivery only after native training, `UpdatePruningStep`, `strip_pruning`, TFLite conversion, output parity, and target-device measurement all pass.

**Failure analysis:** A numerical schedule is not a TensorFlow execution. Installing TensorFlow without a compatible GPU stack may move compute to CPU. A stripped model can retain dense tensors with zeros, and zip compression can be confused with runtime memory savings.


## 10. Extend the evidence

Run the notebook in a pinned TensorFlow/TFMOT environment, retain wrapper and stripped summaries, convert to TFLite, and benchmark the exact target device.

The full evidence boundary and references are in [`README.md`](README.md).
